# Ducati: Guida a Datapizza AI

Questo notebook mostra come utilizzare la libreria `datapizza-ai` per creare chatbot e integrare tool.

## 1. Setup

Assicurati di avere le variabili d'ambiente configurate (es. `OPENAI_API_KEY`).

In [1]:
import os
from dotenv import load_dotenv

load_dotenv() # Carica le variabili d'ambiente dal file .env se presente

True

## 2. Chiamata API Semplice

Ecco come effettuare una semplice invocazione al modello.

In [4]:
from datapizza.clients.openai import OpenAIClient

client = OpenAIClient(
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5.1", # Sostituisci con il modello desiderato
    temperature=1.2
)

response = client.invoke("Raccontami una barzelletta sugli IT in Ducati che fa ridere.")
print(response.text)

Un lunedì mattina al reparto IT Ducati il capo entra furioso:

«Ragazzi, nel weekend qualcuno ha cambiato la password del server di produzione! Chi è stato?»

Silenzio tombale. Dopo un po’ l’ultimo stagista alza la mano timido:

«Capo… sono stato io… ma l’ho fatto per sicurezza.»

Il capo sospira: «Va bene, capita… dimmi la nuova password così sistemiamo.»

E lo stagista:  
«Non posso.»  

«Come non puoi?!»  

«È che… il campo richiedeva almeno 12 caratteri, e io ho scritto:  
`rossiMarquezStonerBaylissCapirossiCheca`  

…e adesso non ricordo dove ho messo il foglietto.»


## 3. Chatbot Semplice

Implementazione di una classe `Chatbot` che gestisce la memoria della conversazione.

In [2]:
from datapizza.memory import Memory
from datapizza.type import ROLE, TextBlock
from datapizza.cache import MemoryCache

class Chatbot:
    def __init__(self, client):
        self.client = client
        self.memory = Memory()

    def send(self, user_input: str) -> str:
        self.memory.add_turn([TextBlock(content=user_input)], ROLE.USER)
        response = self.client.invoke(user_input, memory=self.memory)
        self.memory.add_turn([TextBlock(content=response.text)], ROLE.ASSISTANT)
        total_tokens = (response.prompt_tokens_used or 0) + (response.completion_tokens_used or 0)
        print(f"[metriche] token totali: {total_tokens}")
        return response.text

### Esecuzione del Chatbot
Esegui la cella sottostante per avviare la chat interattiva. Digita 'esci' per fermarla.

In [ ]:
# Configura il client (usa gpt-5 come richiesto o un altro modello disponibile)
client = OpenAIClient(
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5.1", 
    temperature=1
)

bot = Chatbot(client)
print("Chat pronta. Digita 'esci' per terminare.")

while True:
    try:
        user = input("tu> ").strip()
        if user.lower() in {"esci", "exit", "quit"}:
            print("Terminazione chat.")
            break
        if not user:
            continue
        print(user)
        print("bot>", bot.send(user))
    except KeyboardInterrupt:
        print("\nInterrotto dall'utente.")
        break
    except Exception as e:
        print(f"bot> Si è verificato un errore: {e}")
        break

Chat pronta. Digita 'esci' per terminare.
[metriche] token totali: 47
bot> Ciao! Sto bene, grazie. E tu, come stai?
[metriche] token totali: 161
bot> Che bella idea! 😄  
Dove ti piacerebbe andare? 

Ad esempio, posso aiutarti a:
- scegliere una destinazione (Italia o estero)
- trovare il periodo migliore (meno affollato / più economico)
- organizzare viaggio e alloggio
- fare una lista di cosa portare

Dimmi:
1) Da dove parti?  
2) Preferisci spiaggia tranquilla o piena di locali?  
3) Budget più o meno?
Terminazione chat.


## 4. Integrare i Tool con Datapizza-AI

I tool permettono al modello di eseguire azioni esterne. Ecco un esempio di come definirli e utilizzarli.

In [6]:
from datapizza.tools import tool

# Definizione di un tool semplice
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    # Simulazione di una chiamata API meteo
    return f"The weather in {location} is sunny and 72°F"

# Utilizzo del tool
response = client.invoke(
    "What's the weather in New York?",
    tools=[get_weather],
    tool_choice="auto" # Il modello decide se usare il tool
)

# Se il modello decide di usare il tool, datapizza-ai gestisce l'esecuzione (dipende dall'implementazione del client)
# Per vedere l'output:
print(response.text)

# Verifica se sono state fatte chiamate a funzioni (se supportato dal client/risposta)
if hasattr(response, 'function_calls') and response.function_calls:
    for func_call in response.function_calls:
        result = func_call.tool(**func_call.arguments)
        print(f"Tool result ({func_call.name}): {result}")


Tool result (get_weather): The weather in New York is sunny and 72°F


In [7]:
# Esempio con tool multipli
@tool
def calculate(expression: str) -> str:
    """Calculate a mathematical expression."""
    try:
        result = eval(expression)  # Attenzione: eval è pericoloso in produzione
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_time() -> str:
    """Get the current time."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

response = client.invoke(
    "What time is it and what's 15 * 8?",
    tools=[get_time, calculate]
)

print(response.text)

if hasattr(response, 'function_calls') and response.function_calls:
    for func_call in response.function_calls:
        result = func_call.tool(**func_call.arguments)
        print(f"{func_call.name}: {result}")


get_time: 2025-11-28 16:32:32
calculate: 120
